# TALISMAN: Take-home exercise
## Getting started

**First, read `README.md`** for the full task prompts, the time budget, and what to submit. This notebook is a starting point: The first cells show how to use the bench. How to build it / reset, read a camera frame and the sensors, send a command to the two actuators, estimate the laser spot position, and log a run. Rewrite, delete, and reorder freely to complete the tasks of the exercise.

*If you prefer to work with Python scripts and markdown, that's also fine. You can copy existing code from here as necessary.*

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from talisman import LaserEnv

## 1) Build your bench

Paste the token from your invitation email below. A token always builds the
same bench, and `reset()` replays the identical session, so this notebook is
reproducible.

> ⚠️ **Use your own token.** Any string builds a valid bench, so if you
> leave the placeholder in, you would silently work on the wrong one.

In [ ]:
TOKEN = "YOUR_TOKEN"

if TOKEN == "YOUR_TOKEN":
    raise ValueError(
        "Paste your personal token from the invitation email into TOKEN above. "
        "Any string builds a valid bench, so without your own token you would "
        "silently work on the wrong one."
    )

env = LaserEnv(token=TOKEN)
obs, info = env.reset()

## 2) One observation

Each step gives a 64×64 px camera frame and five named scalar sensors. Positions
are `(x, y)` in pixels with `image[y, x]`. Plot with `origin="lower"` so y points
up and markers you overlay at (x, y) land on the matching pixel.

In [ ]:
image = obs["image"]
target = info["target"]  # the fixed setpoint (x, y) in pixels, same every step
print("image shape:", image.shape)
print("sensors:", obs["sensors"])

# Plot the camera image
fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(image, origin="lower", cmap="magma")
fig.colorbar(im, ax=ax, label="counts")
ax.plot(*target, "c+", markersize=14, markeredgewidth=2, label="target")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.legend()
plt.show()

## 3) Sending commands

A command is a length-2 array in [-1, 1] (values outside are clipped). Note that this is a **held setpoint**, not a one-shot nudge: the spot eases towards it only while you keep sending it, and
drifts back when you stop. `step()` returns the [Farama Gymnasium](https://gymnasium.farama.org/)-style 5-tuple *(note that the reward is always 0 and episodes never end)*.

In [ ]:
env.reset()

# One explicit step, so you can see the full return signature:
obs, reward, terminated, truncated, info = env.step(np.zeros(2))

fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(obs["image"], origin="lower", cmap="magma")
fig.colorbar(im, ax=ax, label="counts")
ax.plot(*target, "c+", markersize=14, markeredgewidth=2, label="target")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.set_title("after reset")
ax.legend()
plt.show()

In [ ]:
# Hold the same setpoint for a while and watch the spot settle somewhere new
N_STEPS_HOLD = 30
ACTUATORS = np.array([-1., -1.])

for _ in range(N_STEPS_HOLD):
    obs, *_ = env.step(ACTUATORS)

fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(obs["image"], origin="lower", cmap="magma")
fig.colorbar(im, ax=ax, label="counts")
ax.plot(*target, "c+", markersize=14, markeredgewidth=2, label="target")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.set_title(f"after holding {ACTUATORS} for {N_STEPS_HOLD} steps")
ax.legend()
plt.show()

## 4) A crude position estimate

This is a deliberately-rough baseline for the laser spot position estimate: a background-subtracted centre of mass. It is biased and noisy on purpose. Going beyond it and putting a number on how precisely you know the position will be core Task 1 below.

In [ ]:
def crude_centroid(image):
    """Background-subtracted centre of mass. Crude but functional; going
    beyond this is Task 1. Returns (x, y) in pixels."""
    weights = np.clip(image - np.median(image), 0.0, None)
    total = weights.sum()
    ys, xs = np.indices(image.shape)
    return float((weights * xs).sum() / total), float((weights * ys).sum() / total)


obs, info = env.reset()
cx, cy = crude_centroid(obs["image"])
print(f"crude centroid: ({cx:.2f}, {cy:.2f}) px")

fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(obs["image"], origin="lower", cmap="magma")
fig.colorbar(im, ax=ax, label="counts")
ax.plot(*target, "c+", markersize=14, markeredgewidth=2, label="target")
ax.plot(cx, cy, "bx", markersize=10, label="centroid")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.legend()
plt.show()

## 5) Logging a run

An example loop that steps the bench and collects arrays. It sends the **same fixed `action` on every step** (`action=None` means zero command, i.e. just watching passively). Edit it freely: swap in your own laser spot position estimator, keep the raw camera frames, or send a time-varying command, as needed for the tasks below.

In [ ]:
def log_run(env, n, action=None):
    """Step the bench n times, collecting arrays. `action` is a single
    length-2 command sent UNCHANGED on every one of the n steps (default
    zero = passive watching). For a time-varying command, edit the loop to
    index a per-step sequence instead."""
    action = np.zeros(2) if action is None else np.asarray(action, float)
    env.reset()
    cx, cy = np.empty(n), np.empty(n)
    sensors = np.empty((n, len(env.sensor_names)))
    for i in range(n):
        obs, _, _, _, info = env.step(action)
        cx[i], cy[i] = crude_centroid(obs["image"])
        sensors[i] = list(obs["sensors"].values())
    return dict(t=np.arange(1, n + 1), cx=cx, cy=cy, sensors=sensors)


run = log_run(env, 100, action=None)  # just watch without acting

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(run["t"], run["cx"], label="x")
ax.plot(run["t"], run["cy"], label="y")
ax.axhline(target[0], ls="--", color="C0", alpha=0.5)
ax.axhline(target[1], ls="--", color="C1", alpha=0.5)
ax.set_xlabel("step")
ax.set_ylabel("position (px)")
ax.set_title("passive run with crude centroid (dashed: target)")
ax.legend()
plt.show()

---
# Your investigation starts here

The four core tasks are below as empty slots. Fill them in, add and remove cells freely, and keep a short running log of what you try. The full prompts are in `README.md`.

### Task 1: Measure the laser spot position well

Go beyond `crude_centroid`, and put a number on *how precisely you know the
position* in a single frame.

### Task 2: Watch how the laser spot moves

Leave the system alone (zero actions) for a while and describe what you observe about the spot's motion. Back what you claim with evidence.

### Task 3: Find out what the actuators do

Send deliberate test commands (`log_run(env, n, action=[...])`) and quantify how
each channel moves the laser spot: how far, which direction, how fast.

### Task 4: Look at the telemetry

How are the five channels related to the laser spot's motion, and to each other? Which carry real information about it, and how do you tell the rest apart?

### Investigation log

5–10 bullets, "what I tried, in order", dead ends included.

-

### Control design proposal (words, not code)

A few sentences: what would you feed back on? Which measured signals are worth
adding, and why? How would you handle sudden laser spot displacements? What would you
expect to go wrong on real hardware? And what did you consider but decide
*not* to do?

### A note on AI tools

A sentence or two on how you used AI tools, if any (it doesn't count against you).